In [1]:
!pip install evaluate sacrebleu indic-nlp-library datasets transformers

In [2]:
import torch
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    EncoderDecoderModel,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
from indicnlp.transliterate.unicode_transliterate import UnicodeIndicTransliterator

# Configuration & Hyperparameters
ENCODER_ID = "google-bert/bert-base-uncased"
DECODER_ID = "ai4bharat/indic-bert"
MAX_LENGTH = 128
BATCH_SIZE = 8
LEARNING_RATE = 3e-5
WARMUP_STEPS = 5000
EPOCHS = 3

In [3]:
def preprocess_gujarati_to_devanagari(text):
    """Normalizes Gujarati text and converts the script to Devanagari."""
    # Normalize
    normalizer = IndicNormalizerFactory().get_normalizer("gu")
    normalized_text = normalizer.normalize(text)
    
    # Transliterate Gujarati to Hindi (Devanagari) to share lexical representations
    devanagari_text = UnicodeIndicTransliterator.transliterate(normalized_text, "gu", "hi")
    return devanagari_text

In [4]:
import os

# Search through the input directory and print the exact paths for the en-gu files
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if 'en-gu' in dirname and ('train.en' in filename or 'train.gu' in filename):
            print(os.path.join(dirname, filename))

/kaggle/input/datasets/mathurinache/samanantar/final_data/en-gu/train.gu
/kaggle/input/datasets/mathurinache/samanantar/final_data/en-gu/train.en


In [ ]:
from huggingface_hub import login

# Replace the string below with your actual Hugging Face token
login(token="enter your hugging face token heres")



In [6]:
import os
from datasets import Dataset, DatasetDict

# ==========================================
# 3. Data Loading & Tokenization (Kaggle Samanantar)
# ==========================================

# Define the paths to the Kaggle dataset files
en_file_path = "/kaggle/input/datasets/mathurinache/samanantar/final_data/en-gu/train.en"
gu_file_path = "/kaggle/input/datasets/mathurinache/samanantar/final_data/en-gu/train.gu"

# Read the files line by line
with open(en_file_path, "r", encoding="utf-8") as f:
    # Optional: Add [:100000] at the end of readlines() if you run out of RAM during testing
    english_sentences = [line.strip() for line in f.readlines()]

with open(gu_file_path, "r", encoding="utf-8") as f:
    gujarati_sentences = [line.strip() for line in f.readlines()]

# Ensure both files have the exact same number of lines
assert len(english_sentences) == len(gujarati_sentences), "Mismatched parallel data!"

# Create a Hugging Face Dataset from the lists
raw_dataset = Dataset.from_dict({
    "english": english_sentences,
    "gujarati": gujarati_sentences
})

# Samanantar is massive, so we split 1% (or a smaller fraction) to use as the validation set
# The seed ensures reproducible splits across different kernel runs
dataset = raw_dataset.train_test_split(test_size=0.01, seed=42)

# Rename the 'test' split to 'validation' to match Seq2SeqTrainer's default expectations
dataset = DatasetDict({
    "train": dataset["train"],
    "validation": dataset["test"]
})

tokenizer_enc = AutoTokenizer.from_pretrained(ENCODER_ID)
tokenizer_dec = AutoTokenizer.from_pretrained(DECODER_ID)

def tokenize_function(batch):
    # Tokenize English Source
    inputs = tokenizer_enc(
        batch["english"], 
        padding="max_length", 
        truncation=True, 
        max_length=MAX_LENGTH
    )
    
    # Preprocess and Tokenize Gujarati Target
    guj_texts = [preprocess_gujarati_to_devanagari(text) for text in batch["gujarati"]]
    targets = tokenizer_dec(
        guj_texts, 
        padding="max_length", 
        truncation=True, 
        max_length=MAX_LENGTH
    )
    
    # Assign labels for Seq2Seq loss calculation
    inputs["labels"] = targets["input_ids"]
    return inputs

# Map the tokenization function across the newly created dataset
# Note: For massive datasets, removing 'batched=True' or setting a batch_size might be necessary depending on RAM
tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/3023730 [00:00<?, ? examples/s]

KeyboardInterrupt: 

In [ ]:
# Map the tokenization function
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Save it to Kaggle's output directory
tokenized_datasets.save_to_disk("/kaggle/working/tokenized_samanantar_en_gu")
print("Dataset saved to disk successfully!")

In [ ]:
import shutil
from IPython.display import FileLink

# 1. Zip the saved dataset folder
print("Zipping the dataset...")
shutil.make_archive('/kaggle/working/tokenized_dataset', 'zip', '/kaggle/working/tokenized_samanantar_en_gu')
print("Done!")

# 2. Create a clickable link to download the zip file
FileLink(r'tokenized_dataset.zip')

In [ ]:
'''
import torch
import torch.nn as nn

class SingleAttentionHead(nn.Module):
    def __init__(self, query_key_embedding_dim, value_embedding_dim, sha_dim, masked, is_dropout, dropout_probability):
        super().__init__()
        self.sha_dim = sha_dim
        self.masked = masked
        self.is_dropout = is_dropout

        self.query_projection_layer = nn.Linear(query_key_embedding_dim, sha_dim, bias=False)
        self.key_projection_layer = nn.Linear(query_key_embedding_dim, sha_dim, bias=False)
        self.value_projection_layer = nn.Linear(value_embedding_dim, sha_dim, bias=False)
        
        if self.is_dropout:
            self.single_head_attn_mask_dropout = nn.Dropout(p=dropout_probability)
            
        self.softmax_activation = nn.Softmax(dim=-1) # Fixed dim=-1 for correct attention weighting

    def forward(self, query_embedding, key_embedding, value_embedding):
        projected_query = self.query_projection_layer(query_embedding)
        projected_key = self.key_projection_layer(key_embedding)
        projected_value = self.value_projection_layer(value_embedding)

        # Fixed transpose to support batching (-2, -1 swaps the last two dimensions)
        query_key_similarity = torch.matmul(projected_query, torch.transpose(projected_key, -2, -1)) / math.sqrt(self.sha_dim)

        if self.masked:
            # Create a lower triangular mask to prevent looking at future words
            mask = torch.tril(torch.ones_like(query_key_similarity))
            query_key_similarity = query_key_similarity.masked_fill(mask == 0, float('-inf'))
            
        query_key_soft = self.softmax_activation(query_key_similarity)

        if self.is_dropout:
            query_key_soft = self.single_head_attn_mask_dropout(query_key_soft)
            
        weighted_attn_embedding = torch.matmul(query_key_soft, projected_value)
        return weighted_attn_embedding

class MultiHeadAttentionLayer(nn.Module):
    def __init__(self, query_key_embedding_dim, value_embedding_dim, num_attn_heads, masked, is_dropout, dropout_probability):
        super().__init__()
        sha_dim = value_embedding_dim // num_attn_heads
        
        self.attn_heads = nn.ModuleList()

        self.mha_projection_layer = nn.Linear(value_embedding_dim, value_embedding_dim, bias=False)
        self.is_dropout = is_dropout
        if self.is_dropout:
            self.mha_dropout_layer = nn.Dropout(p=dropout_probability) 

    def forward(self, query_embedding, key_embedding, value_embedding):
        attn_heads_weighted_embeddings = [
            head(query_embedding, key_embedding, value_embedding) for head in self.attn_heads
        ]
        mha_concatenated_embeddings = torch.cat(attn_heads_weighted_embeddings, dim=-1)
        mha_output = self.mha_projection_layer(mha_concatenated_embeddings)

        if self.is_dropout:
            mha_output = self.mha_dropout_layer(mha_output)
        return mha_output

class EncoderLayer(nn.Module):
    def __init__(self, model_dim, num_attn_heads, is_dropout, dropout_probability, ffn_projection_dim):
        super().__init__()
        self.mha_layer = MultiHeadAttentionLayer(model_dim, model_dim, num_attn_heads, False, is_dropout, dropout_probability)
        self.first_layer_norm = nn.LayerNorm(model_dim)
        self.dropout_layer = nn.Dropout(p=dropout_probability) if is_dropout else nn.Identity()
            
        self.ffn = nn.Sequential(
            nn.Linear(model_dim, ffn_projection_dim),
            nn.GELU(),
            nn.Linear(ffn_projection_dim, model_dim)
        )
        self.second_layer_norm = nn.LayerNorm(model_dim)
    
    def forward(self, x):
        mha_out = self.mha_layer(x, x, x)
        x = self.first_layer_norm(x + self.dropout_layer(mha_out))
        ffn_out = self.ffn(x)
        x = self.second_layer_norm(x + self.dropout_layer(ffn_out))
        return x

class DecoderLayer(nn.Module):
    def __init__(self, model_dim, num_attn_heads, is_dropout, dropout_probability, ffn_projection_dim):
        super().__init__()
        # 1. Masked Self-Attention (Gujarati words can only look at past Gujarati words)
        self.masked_mha_layer = MultiHeadAttentionLayer(model_dim, model_dim, num_attn_heads, True, is_dropout, dropout_probability)
        self.first_layer_norm = nn.LayerNorm(model_dim)
        
        # 2. Cross-Attention (Gujarati words looking back at the English Encoder output)
        self.cross_mha_layer = MultiHeadAttentionLayer(model_dim, model_dim, num_attn_heads, False, is_dropout, dropout_probability)
        self.second_layer_norm = nn.LayerNorm(model_dim)

        self.ffn = nn.Sequential(
            nn.Linear(model_dim, ffn_projection_dim),
            nn.GELU(),
            nn.Linear(ffn_projection_dim, model_dim)
        )
        self.third_layer_norm = nn.LayerNorm(model_dim)
        self.dropout_layer = nn.Dropout(p=dropout_probability) if is_dropout else nn.Identity()

    def forward(self, tgt_x, encoder_output):
        masked_mha_out = self.masked_mha_layer(tgt_x, tgt_x, tgt_x)
        tgt_x = self.first_layer_norm(tgt_x + self.dropout_layer(masked_mha_out))
        
        cross_mha_out = self.cross_mha_layer(query_embedding=tgt_x, key_embedding=encoder_output, value_embedding=encoder_output)
        tgt_x = self.second_layer_norm(tgt_x + self.dropout_layer(cross_mha_out))
        
        ffn_out = self.ffn(tgt_x)
        tgt_x = self.third_layer_norm(tgt_x + self.dropout_layer(ffn_out))
        return tgt_x

class CustomTranslationModel(nn.Module):
    def __init__(self, source_vocab_size, target_vocab_size, context_len, model_dim, num_layers, num_attn_heads, dropout_prob):
        super().__init__()
        # Token & Position Embeddings
        self.src_embedding = nn.Embedding(source_vocab_size, model_dim)
        self.tgt_embedding = nn.Embedding(target_vocab_size, model_dim)
        self.pos_encoding = nn.Embedding(context_len, model_dim)
        self.dropout = nn.Dropout(p=dropout_prob)

        # Stacks
        self.encoder_stack = nn.ModuleList()
        self.decoder_stack = nn.ModuleList()
        
        # Output layer to predict Gujarati tokens
        self.lm_head = nn.Linear(model_dim, target_vocab_size)

    def forward(self, src_tokens, tgt_tokens):
        # Build position tensors
        batch_size, src_seq_len = src_tokens.size()
        tgt_seq_len = tgt_tokens.size(1)
        
        src_positions = torch.arange(0, src_seq_len, device=src_tokens.device).unsqueeze(0).expand(batch_size, src_seq_len)
        tgt_positions = torch.arange(0, tgt_seq_len, device=tgt_tokens.device).unsqueeze(0).expand(batch_size, tgt_seq_len)
        
        # 1. Encode English
        enc_out = self.dropout(self.src_embedding(src_tokens) + self.pos_encoding(src_positions))
        for layer in self.encoder_stack:
            enc_out = layer(enc_out)
            
        # 2. Decode Gujarati
        dec_out = self.dropout(self.tgt_embedding(tgt_tokens) + self.pos_encoding(tgt_positions))
        for layer in self.decoder_stack:
            dec_out = layer(dec_out, enc_out)
            
        # 3. Predict Next Token
        logits = self.lm_head(dec_out)
        return logits
        '''
        #in next block updated single head attemtion

In [ ]:
'''
import math
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# --- 1. Prepare DataLoaders ---
# Convert Hugging Face datasets to PyTorch tensors
tokenized_datasets.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

train_dataloader = DataLoader(tokenized_datasets["train"], batch_size=8, shuffle=True)
val_dataloader = DataLoader(tokenized_datasets["validation"], batch_size=8)

# --- 2. Initialize the Custom Model ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

# We use standard hyperparameters for a "Base" size transformer
custom_model = CustomTranslationModel(
    source_vocab_size=tokenizer_enc.vocab_size,
    target_vocab_size=tokenizer_dec.vocab_size,
    context_len=MAX_LENGTH,
    model_dim=256,       # Hidden size
    num_layers=4,        # Number of encoder/decoder blocks
    num_attn_heads=8,    # Number of attention heads
    dropout_prob=0.1
).to(device)

# --- 3. Setup Optimizer and Loss ---
optimizer = torch.optim.AdamW(custom_model.parameters(), lr=1e-4)

# We use CrossEntropyLoss but tell it to completely ignore the padding tokens
pad_idx = tokenizer_dec.pad_token_id
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

# --- 4. The Training Loop ---
NUM_EPOCHS = 5

for epoch in range(NUM_EPOCHS):
    custom_model.train()
    total_train_loss = 0
    
    # Wrap dataloader in tqdm for a progress bar
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    
    for batch in progress_bar:
        optimizer.zero_grad()
        
        # English source
        src_tokens = batch["input_ids"].to(device)
        # Gujarati target
        tgt_tokens = batch["labels"].to(device)
        
        # Teacher Forcing: 
        # The model sees Gujarati up to the current word (decoder_input) 
        # and has to predict the NEXT word (shifted_labels)
        decoder_input = tgt_tokens[:, :-1]
        shifted_labels = tgt_tokens[:, 1:]
        
        # Forward pass through our custom network
        logits = custom_model(src_tokens, decoder_input)
        
        # Calculate loss (flatten the batch and sequence length dimensions together)
        loss = criterion(logits.reshape(-1, tokenizer_dec.vocab_size), shifted_labels.reshape(-1))
        
        # Backpropagation
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item()
        progress_bar.set_postfix({"loss": loss.item()})
        
    avg_train_loss = total_train_loss / len(train_dataloader)
    print(f"Epoch {epoch+1} Completed | Average Training Loss: {avg_train_loss:.4f}")
    '''
#slow as f

In [7]:

import torch
import torch.nn as nn

import torch.nn.functional as F

class SingleAttentionHead(nn.Module):
    def __init__(self, query_key_embedding_dim, value_embedding_dim, sha_dim, masked, is_dropout, dropout_probability):
        super().__init__()
        self.sha_dim = sha_dim
        self.masked = masked
        self.is_dropout = is_dropout
        self.dropout_probability = dropout_probability

        self.query_projection_layer = nn.Linear(query_key_embedding_dim, sha_dim, bias=False)
        self.key_projection_layer = nn.Linear(query_key_embedding_dim, sha_dim, bias=False)
        self.value_projection_layer = nn.Linear(value_embedding_dim, sha_dim, bias=False)

    def forward(self, query_embedding, key_embedding, value_embedding):
        projected_query = self.query_projection_layer(query_embedding)
        projected_key = self.key_projection_layer(key_embedding)
        projected_value = self.value_projection_layer(value_embedding)

        # Use PyTorch's highly optimized Scaled Dot Product Attention (FlashAttention)
        # This completely replaces the manual torch.matmul and Softmax steps
        dropout_p = self.dropout_probability if self.is_dropout and self.training else 0.0
        
        weighted_attn_embedding = F.scaled_dot_product_attention(
            query=projected_query,
            key=projected_key,
            value=projected_value,
            dropout_p=dropout_p,
            is_causal=self.masked # Automatically applies the triangular mask if True
        )

        return weighted_attn_embedding
 

class MultiHeadAttentionLayer(nn.Module):
    def __init__(self, query_key_embedding_dim, value_embedding_dim, num_attn_heads, masked, is_dropout, dropout_probability):
        super().__init__()
        sha_dim = value_embedding_dim // num_attn_heads
        
        self.attn_heads = nn.ModuleList()

        self.mha_projection_layer = nn.Linear(value_embedding_dim, value_embedding_dim, bias=False)
        self.is_dropout = is_dropout
        if self.is_dropout:
            self.mha_dropout_layer = nn.Dropout(p=dropout_probability) 

    def forward(self, query_embedding, key_embedding, value_embedding):
        attn_heads_weighted_embeddings = [
            head(query_embedding, key_embedding, value_embedding) for head in self.attn_heads
        ]
        mha_concatenated_embeddings = torch.cat(attn_heads_weighted_embeddings, dim=-1)
        mha_output = self.mha_projection_layer(mha_concatenated_embeddings)

        if self.is_dropout:
            mha_output = self.mha_dropout_layer(mha_output)
        return mha_output

class EncoderLayer(nn.Module):
    def __init__(self, model_dim, num_attn_heads, is_dropout, dropout_probability, ffn_projection_dim):
        super().__init__()
        self.mha_layer = MultiHeadAttentionLayer(model_dim, model_dim, num_attn_heads, False, is_dropout, dropout_probability)
        self.first_layer_norm = nn.LayerNorm(model_dim)
        self.dropout_layer = nn.Dropout(p=dropout_probability) if is_dropout else nn.Identity()
            
        self.ffn = nn.Sequential(
            nn.Linear(model_dim, ffn_projection_dim),
            nn.GELU(),
            nn.Linear(ffn_projection_dim, model_dim)
        )
        self.second_layer_norm = nn.LayerNorm(model_dim)
    
    def forward(self, x):
        mha_out = self.mha_layer(x, x, x)
        x = self.first_layer_norm(x + self.dropout_layer(mha_out))
        ffn_out = self.ffn(x)
        x = self.second_layer_norm(x + self.dropout_layer(ffn_out))
        return x

class DecoderLayer(nn.Module):
    def __init__(self, model_dim, num_attn_heads, is_dropout, dropout_probability, ffn_projection_dim):
        super().__init__()
        # 1. Masked Self-Attention (Gujarati words can only look at past Gujarati words)
        self.masked_mha_layer = MultiHeadAttentionLayer(model_dim, model_dim, num_attn_heads, True, is_dropout, dropout_probability)
        self.first_layer_norm = nn.LayerNorm(model_dim)
        
        # 2. Cross-Attention (Gujarati words looking back at the English Encoder output)
        self.cross_mha_layer = MultiHeadAttentionLayer(model_dim, model_dim, num_attn_heads, False, is_dropout, dropout_probability)
        self.second_layer_norm = nn.LayerNorm(model_dim)

        self.ffn = nn.Sequential(
            nn.Linear(model_dim, ffn_projection_dim),
            nn.GELU(),
            nn.Linear(ffn_projection_dim, model_dim)
        )
        self.third_layer_norm = nn.LayerNorm(model_dim)
        self.dropout_layer = nn.Dropout(p=dropout_probability) if is_dropout else nn.Identity()

    def forward(self, tgt_x, encoder_output):
        masked_mha_out = self.masked_mha_layer(tgt_x, tgt_x, tgt_x)
        tgt_x = self.first_layer_norm(tgt_x + self.dropout_layer(masked_mha_out))
        
        cross_mha_out = self.cross_mha_layer(query_embedding=tgt_x, key_embedding=encoder_output, value_embedding=encoder_output)
        tgt_x = self.second_layer_norm(tgt_x + self.dropout_layer(cross_mha_out))
        
        ffn_out = self.ffn(tgt_x)
        tgt_x = self.third_layer_norm(tgt_x + self.dropout_layer(ffn_out))
        return tgt_x

class CustomTranslationModel(nn.Module):
    def __init__(self, source_vocab_size, target_vocab_size, context_len, model_dim, num_layers, num_attn_heads, dropout_prob):
        super().__init__()
        # Token & Position Embeddings
        self.src_embedding = nn.Embedding(source_vocab_size, model_dim)
        self.tgt_embedding = nn.Embedding(target_vocab_size, model_dim)
        self.pos_encoding = nn.Embedding(context_len, model_dim)
        self.dropout = nn.Dropout(p=dropout_prob)

        # Stacks
        self.encoder_stack = nn.ModuleList()
        self.decoder_stack = nn.ModuleList()
        
        # Output layer to predict Gujarati tokens
        self.lm_head = nn.Linear(model_dim, target_vocab_size)

    def forward(self, src_tokens, tgt_tokens):
        # Build position tensors
        batch_size, src_seq_len = src_tokens.size()
        tgt_seq_len = tgt_tokens.size(1)
        
        src_positions = torch.arange(0, src_seq_len, device=src_tokens.device).unsqueeze(0).expand(batch_size, src_seq_len)
        tgt_positions = torch.arange(0, tgt_seq_len, device=tgt_tokens.device).unsqueeze(0).expand(batch_size, tgt_seq_len)
        
        # 1. Encode English
        enc_out = self.dropout(self.src_embedding(src_tokens) + self.pos_encoding(src_positions))
        for layer in self.encoder_stack:
            enc_out = layer(enc_out)
            
        # 2. Decode Gujarati
        dec_out = self.dropout(self.tgt_embedding(tgt_tokens) + self.pos_encoding(tgt_positions))
        for layer in self.decoder_stack:
            dec_out = layer(dec_out, enc_out)
            
        # 3. Predict Next Token
        logits = self.lm_head(dec_out)
        return logits
        
        #updated single head attemtion

In [8]:
import os

# Search your inputs for the Hugging Face dataset configuration file
for dirname, _, filenames in os.walk('/kaggle/input'):
    if 'dataset_dict.json' in filenames:
        print("Your tokenized dataset is located at:", dirname)

Your tokenized dataset is located at: /kaggle/input/datasets/echoesinpages/tokenized-dataset


In [9]:
from datasets import load_from_disk

# Paste the exact path printed by the search script
DATASET_PATH = "/kaggle/input/datasets/echoesinpages/tokenized-dataset"

tokenized_datasets = load_from_disk(DATASET_PATH)
print("Tokenized dataset loaded from disk successfully!")

Tokenized dataset loaded from disk successfully!


In [12]:

import math
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# --- 1. Prepare DataLoaders (Optimized) ---
tokenized_datasets.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# Added num_workers=2 and pin_memory=True to speed up data transfer to the GPU
train_dataloader = DataLoader(
    tokenized_datasets["train"], 
    batch_size=16, 
    shuffle=True,
    num_workers=2,
    pin_memory=True 
)
val_dataloader = DataLoader(tokenized_datasets["validation"], batch_size=16, num_workers=2, pin_memory=True)

# --- 2. Initialize the Custom Model ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

custom_model = CustomTranslationModel(
    source_vocab_size=tokenizer_enc.vocab_size,
    target_vocab_size=tokenizer_dec.vocab_size,
    context_len=MAX_LENGTH,
    model_dim=256,       
    num_layers=4,        
    num_attn_heads=8,    
    dropout_prob=0.1
).to(device)

# --- 3. Setup Optimizer, Loss, and AMP Scaler ---
optimizer = torch.optim.AdamW(custom_model.parameters(), lr=1e-4)
pad_idx = tokenizer_dec.pad_token_id
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

# Initialize the gradient scaler for Automatic Mixed Precision (AMP)
scaler = torch.amp.GradScaler('cuda')

# --- 4. The Optimized Training Loop ---
NUM_EPOCHS = 3

for epoch in range(NUM_EPOCHS):
    custom_model.train()
    total_train_loss = 0
    
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    
    for batch in progress_bar:
        # set_to_none=True is slightly faster than traditional zero_grad()
        optimizer.zero_grad(set_to_none=True)
        
        src_tokens = batch["input_ids"].to(device, non_blocking=True)
        tgt_tokens = batch["labels"].to(device, non_blocking=True)
        
        decoder_input = tgt_tokens[:, :-1]
        shifted_labels = tgt_tokens[:, 1:]
        
        # Enables Automatic Mixed Precision (FP16) for lightning-fast forward passes
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits = custom_model(src_tokens, decoder_input)
            loss = criterion(logits.reshape(-1, tokenizer_dec.vocab_size), shifted_labels.reshape(-1))
        
        # Backpropagate using the scaler
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_train_loss += loss.item()
        progress_bar.set_postfix({"loss": loss.item()})
        
    avg_train_loss = total_train_loss / len(train_dataloader)
    print(f"Epoch {epoch+1} Completed | Average Training Loss: {avg_train_loss:.4f}")
    

Training on device: cuda


Epoch 1/3:   0%|          | 0/188984 [00:00<?, ?it/s]

/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [352,0,0], thread: [32,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [352,0,0], thread: [33,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [352,0,0], thread: [34,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [352,0,0], thread: [35,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [352,0,0], thread: [36,0,0] 

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [13]:
import os
import torch
import torch.nn as nn
from tqdm.auto import tqdm

# --- 1. Hyperparameters & Setup (Adjusted for absolute VRAM safety) ---
LEARNING_RATE = 1e-4
NUM_EPOCHS = 3
BATCH_SIZE = 8                  # Dropped to 8 to guarantee it fits in 15GB VRAM
GRAD_ACCUMULATION_STEPS = 16    # Increased to 16 to maintain the 128 effective batch size
SAVE_EVERY_N_STEPS = 5000       

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pad_idx = tokenizer_dec.pad_token_id
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
scaler = torch.amp.GradScaler('cuda')

# --- 2. Initialize Model & Optimizer ---
print(f"Initializing model on device: {device}...")
custom_model = CustomTranslationModel(
    source_vocab_size=tokenizer_enc.vocab_size,
    target_vocab_size=tokenizer_dec.vocab_size,
    context_len=MAX_LENGTH,
    model_dim=256,       
    num_layers=4,        
    num_attn_heads=8,    
    dropout_prob=0.1
).to(device)

optimizer = torch.optim.AdamW(custom_model.parameters(), lr=LEARNING_RATE)

# --- 3. Advanced Checkpointing ---
START_EPOCH = 0
START_STEP = 0
checkpoint_path = "/kaggle/working/translation_checkpoint.pt"

if os.path.exists(checkpoint_path):
    print(f"Found checkpoint at {checkpoint_path}. Loading weights...")
    checkpoint = torch.load(checkpoint_path)
    custom_model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    START_EPOCH = checkpoint['epoch']
    START_STEP = checkpoint['step']
    print(f"Resuming precisely from Epoch {START_EPOCH + 1}, Batch {START_STEP}...")
else:
    print("No checkpoint found. Starting from scratch...")

# --- 4. The High-Efficiency Training Loop (No Compiler) ---
print("Starting training loop...")
for epoch in range(START_EPOCH, NUM_EPOCHS):
    custom_model.train() # Using the raw model directly
    total_train_loss = 0
    
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    
    optimizer.zero_grad(set_to_none=True)
    
    for step, batch in enumerate(progress_bar):
        if step < START_STEP:
            continue
            
        src_tokens = batch["input_ids"].to(device, non_blocking=True)
        tgt_tokens = batch["labels"].to(device, non_blocking=True)
        
        decoder_input = tgt_tokens[:, :-1]
        shifted_labels = tgt_tokens[:, 1:]
        
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits = custom_model(src_tokens, decoder_input)
            loss = criterion(logits.reshape(-1, tokenizer_dec.vocab_size), shifted_labels.reshape(-1))
            loss = loss / GRAD_ACCUMULATION_STEPS
        
        scaler.scale(loss).backward()
        
        if ((step + 1) % GRAD_ACCUMULATION_STEPS == 0) or ((step + 1) == len(train_dataloader)):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
        
        total_train_loss += (loss.item() * GRAD_ACCUMULATION_STEPS)
        progress_bar.set_postfix({"loss": loss.item() * GRAD_ACCUMULATION_STEPS})
        
        if step > 0 and step % SAVE_EVERY_N_STEPS == 0:
            torch.save({
                'epoch': epoch,
                'step': step + 1, 
                'model_state_dict': custom_model.state_dict(), 
                'optimizer_state_dict': optimizer.state_dict(),
                'scaler_state_dict': scaler.state_dict(),
            }, checkpoint_path)
            
    avg_train_loss = total_train_loss / (len(train_dataloader) - START_STEP)
    print(f"Epoch {epoch+1} Completed | Average Training Loss: {avg_train_loss:.4f}")
    
    START_STEP = 0 
    torch.save({
        'epoch': epoch + 1,
        'step': 0,
        'model_state_dict': custom_model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
    }, checkpoint_path)
    print(f"Epoch {epoch+1} checkpoint successfully saved!")

Initializing model on device: cuda...


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
